# Setup

In [1]:
# install personal package for use below, type the below into the docker terminal
# pip install /tf/pyGroupSequentialDesigns/

In [2]:
# higher resolution graphs
%config InlineBackend.figure_format='retina'

## Published package imports

In [3]:
# imports for study design step (Step 1)
import numpy as np
import pandas as pd
from scipy import stats
from scipy import optimize

# imports for GP regression (Step 3)
import gpflow

# imports for Bayes opt (Step 4-6)
import trieste
from trieste.space import Box
from trieste.models.gpflow.models import GaussianProcessRegression
import tensorflow as tf
from trieste.experimental.plotting import plot_regret

/usr/local/lib/python3.11/dist-packages/gpflow/versions.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


## Personal package imports

In [4]:
from py_group_sequential_designs import boundaries as bd
from py_group_sequential_designs import feasibility_penalty as fp
from py_group_sequential_designs import format_boundaries_after_ask as fmt_bd
from py_group_sequential_designs import function_to_minimize as fn_min
from py_group_sequential_designs import generate_gpr_input as gen_input
from py_group_sequential_designs import simulate as sim
from py_group_sequential_designs import sample_size as ss

# Bayesian optimization workflow default values

In [5]:
# some set defaults
num_analyses = 3
target_alpha = 0.05
target_power = 0.9
important_diff_delta = 1
assumed_variance = 3

# to obtain mu (sample size at one stage)
mu = ss.sample_size_means(
    ratio=1,
    variance=assumed_variance,
    power=target_power,
    alpha=target_alpha,
    delta=important_diff_delta
)

# Simulate initial points for GPR

## Create a helper function

In [6]:
# create a function that generates the points x and y that will be
# included in the design matrix X and Y
def generate_x_y(
        mu,
        upper_bounds,
        lower_bounds,
        n_analyses,
        target_power,
        target_alpha,
        alpha_prime,
        beta_prime,
        n_power09):
    
    # 2. Generate the GPR input values
    # note that the input includes the sample size at power 0.9
    x = gen_input.generate_gpr_input(
        n_analyses = n_analyses,
        upper_bounds=upper_bounds,
        lower_bounds=lower_bounds,
        n_patients=n_power09)
    
    # 3. Generate maximum expected sample size and feasibility penalty
    max_ess_new = ss.max_ess(
        n_analyses=n_analyses,
        upper_bounds=upper_bounds,
        lower_bounds=lower_bounds,
        n_patients=n_power09)
    
    penalty = fp.feasibility_penalty(
        mu = mu,
        power=target_power,
        alpha=target_alpha,
        beta_prime=beta_prime,
        alpha_prime=alpha_prime
    )
    
    # 4. Calculate the function value (GPR output)
    y = fn_min.function_to_minimize(max_ess_val=max_ess_new, penalty=penalty)

    return (np.array([x]), np.array([[y]]))

## Point 1

In [7]:
# simulate the trial design 
poc_simulation = bd.calculate_pocock_boundaries(
    n_analyses=num_analyses,
    alpha=target_alpha,
    n_patients=20
)

# find the number of patients that achieves 90% power (beta 0.1)
# here we get beta_prime
poc_n_power09, poc_beta_prime = ss.find_sample_size(
    n_analyses = num_analyses,
    upper_bounds = poc_simulation[0],
    lower_bounds = poc_simulation[1],
    alt_hypothesis = important_diff_delta,
    variance = assumed_variance
)

x1, y1 = generate_x_y(
    mu = mu,
    upper_bounds = poc_simulation[0],
    lower_bounds = poc_simulation[1],
    n_analyses = num_analyses,
    target_power = target_power,
    target_alpha = target_alpha,
    alpha_prime = poc_simulation[3],
    beta_prime = poc_beta_prime,
    n_power09 = poc_n_power09
)

## Point 2

In [8]:
# simulate the trial design 
of_simulation = bd.calculate_of_boundaries(
    n_analyses=num_analyses,
    alpha=target_alpha,
    n_patients=20
)

# find the number of patients that achieves 90% power (beta 0.1) and beta_prime
of_n_power09, of_beta_prime = ss.find_sample_size(
    n_analyses = num_analyses,
    upper_bounds = of_simulation[0],
    lower_bounds = of_simulation[1],
    alt_hypothesis = important_diff_delta,
    variance = assumed_variance
)

x2, y2 = generate_x_y(
    mu = mu,
    upper_bounds = of_simulation[0],
    lower_bounds = of_simulation[1],
    n_analyses = num_analyses,
    target_power = target_power,
    target_alpha = target_alpha,
    alpha_prime = of_simulation[3],
    beta_prime = of_beta_prime,
    n_power09 = of_n_power09
)

## Point 3

In [9]:
# simulate the trial design 
tri_simulation = bd.calculate_triangular_boundaries(
    n_analyses=num_analyses,
    alpha=target_alpha,
    delta=important_diff_delta,
    n_patients=20
)

# find the number of patients that achieves 90% power (beta 0.1) and beta_prime
tri_n_power09, tri_beta_prime = ss.find_sample_size(
    n_analyses = num_analyses,
    upper_bounds = tri_simulation[0],
    lower_bounds = tri_simulation[1],
    alt_hypothesis = important_diff_delta,
    variance = assumed_variance
)

x3, y3 = generate_x_y(
    mu = mu,
    upper_bounds = tri_simulation[0],
    lower_bounds = tri_simulation[1],
    n_analyses = num_analyses,
    target_power = target_power,
    target_alpha = target_alpha,
    alpha_prime = tri_simulation[3],
    beta_prime = tri_beta_prime,
    n_power09 = tri_n_power09
)

# Bayesian optimization loop

## Enter the initial points

In [10]:
design_matrix = np.concatenate((x1, x2, x3))
design_matrix

array([[ 1.99218511e+00,  1.99218511e+00,  1.99218511e+00,
        -1.99218511e+00, -1.99218511e+00,  1.99629070e+01],
       [ 2.96112400e+00,  2.09383086e+00,  1.70960574e+00,
        -2.96112400e+00, -2.09383086e+00,  1.75540749e+01],
       [ 2.11957748e+00,  1.87345951e+00,  1.83560794e+00,
         6.28553399e-16,  1.12407571e+00,  2.13211892e+01]])

In [11]:
output_vals = np.concatenate((y1, y2, y3))
output_vals

array([[194.44127199],
       [188.97218955],
       [182.4897155 ]])

## Center and scale

In [12]:
def normalize_forward(data, mean=None, std=None):
    if (mean is None) and (std is None):
        mean = np.mean(data)
        std = np.std(data)
    
    return (
        (data - mean) / std,
        mean,
        std
    )

In [13]:
def normalize_backward(norm_data, mean, std):
    return (norm_data * std) + mean

### Test the functions

In [14]:
test_data, mean, std = normalize_forward(design_matrix)
test_data

array([[-0.25657046, -0.25657046, -0.25657046, -0.80367298, -0.80367298,
         2.21102834],
       [-0.12352336, -0.24261326, -0.29537205, -0.93672008, -0.81763018,
         1.88026639],
       [-0.23907794, -0.27287293, -0.27807041, -0.53012172, -0.37577245,
         2.39753702]])

In [15]:
normalize_backward(test_data, mean, std)

array([[ 1.99218511e+00,  1.99218511e+00,  1.99218511e+00,
        -1.99218511e+00, -1.99218511e+00,  1.99629070e+01],
       [ 2.96112400e+00,  2.09383086e+00,  1.70960574e+00,
        -2.96112400e+00, -2.09383086e+00,  1.75540749e+01],
       [ 2.11957748e+00,  1.87345951e+00,  1.83560794e+00,
         4.44089210e-16,  1.12407571e+00,  2.13211892e+01]])

## Normalize and loop

In [16]:
normed_design_matix, design_matrix_mean, design_matrix_std = normalize_forward(design_matrix)

In [17]:
normed_output_vals, output_vals_mean, output_vals_std = normalize_forward(output_vals)

In [18]:
def build_model(X, Y):
    
    kernel = gpflow.kernels.SquaredExponential()

    likelihood = gpflow.likelihoods.Gaussian()
        
    gpr = gpflow.models.GPR(
        data = (X, Y),
        kernel = kernel,
        likelihood = likelihood
    )

    gpflow.utilities.print_summary(gpr, fmt="notebook")
    
    return GaussianProcessRegression(gpr)

In [19]:
bayes_opt_model = build_model(X = normed_design_matix, Y = normed_output_vals)

name,class,transform,prior,trainable,shape,dtype,value
GPR.kernel.variance,Parameter,Softplus,,True,(),float64,1
GPR.kernel.lengthscales,Parameter,Softplus,,True,(),float64,1
GPR.likelihood.variance,Parameter,Softplus + Shift,,True,(),float64,1


In [20]:
# create a dataset that works well with trieste
initial_data = trieste.data.Dataset(
    query_points = normed_design_matix, 
    observations = normed_output_vals
)

In [21]:
# normalize the search space
x_search_space = [-20, -20, -20, 20, 20]
y_search_space = 1000

In [22]:
normalize_forward(x_search_space,
                  design_matrix_mean,
                  design_matrix_std)

(array([-3.27636511, -3.27636511, -3.27636511,  2.21612167,  2.21612167]),
 3.8607045853851583,
 7.282675703947315)

In [23]:
normalize_forward(y_search_space,
                  output_vals_mean,
                  output_vals_std)

(166.09171608785786, 188.63439234594273, 4.885045604711963)

In [24]:
# create the search space using trieste Box function
search_space = Box(
    lower = [-4, -4, -4, -4, -4, 2], 
    upper = [3, 3, 3, 3, 3, 200]
)

In [25]:
initial_data

Dataset(query_points=array([[-0.25657046, -0.25657046, -0.25657046, -0.80367298, -0.80367298,
         2.21102834],
       [-0.12352336, -0.24261326, -0.29537205, -0.93672008, -0.81763018,
         1.88026639],
       [-0.23907794, -0.27287293, -0.27807041, -0.53012172, -0.37577245,
         2.39753702]]), observations=array([[ 1.18870531],
       [ 0.06914924],
       [-1.25785455]]))

In [26]:
ask_tell = trieste.ask_tell_optimization.AskTellOptimizer(
    search_space = search_space,
    datasets = initial_data,
    models = bayes_opt_model,
    acquisition_rule = trieste.acquisition.rule.EfficientGlobalOptimization(
        optimizer = trieste.acquisition.optimizer.generate_continuous_optimizer(
            num_optimization_runs = 5000
        )
    )
)

In [27]:
num_repeats = 50

for i in range(num_repeats):
    normed_results = ask_tell.ask()

    unnormed_results_x = normalize_backward(
        normed_results,
        design_matrix_mean,
        design_matrix_std
    )    
    
    unnormed_new_inputs = fmt_bd.format_boundaries_after_ask(
        n_analyses = num_analyses,
        result = unnormed_results_x
    )

    new_sim_trial = sim.group_sequential_designs(
        n_analyses = num_analyses,
        upper_bounds = unnormed_new_inputs[0],
        lower_bounds = unnormed_new_inputs[1],
        n_patients = unnormed_new_inputs[2],
        null_hypothesis = 0,
        alt_hypothesis = important_diff_delta,
        variance = assumed_variance
    )

    new_x, new_y = generate_x_y(
        mu = mu,
        upper_bounds = unnormed_new_inputs[0],
        lower_bounds= unnormed_new_inputs[1],
        n_analyses = num_analyses,
        target_power = target_power,
        target_alpha = target_alpha,
        alpha_prime = new_sim_trial[1],
        beta_prime = new_sim_trial[2],
        n_power09 = unnormed_new_inputs[2]
    )

    normed_new_x, _, _ = normalize_forward(
        new_x,
        design_matrix_mean,
        design_matrix_std
    )

    normed_new_y, _, _ = normalize_forward(
        new_y,
        output_vals_mean,
        output_vals_std
    )

    new_data = trieste.data.Dataset(
        query_points = normed_new_x, 
        observations = normed_new_y
    )

    ask_tell.tell(new_data=new_data)

In [28]:
ask_tell.to_result()

OptimizationResult(final_result=Ok(Record(datasets={'OBJECTIVE': Dataset(query_points=<tf.Tensor: shape=(53, 6), dtype=float64, numpy=
array([[-2.56570462e-01, -2.56570462e-01, -2.56570462e-01,
        -8.03672982e-01, -8.03672982e-01,  2.21102834e+00],
       [-1.23523362e-01, -2.42613264e-01, -2.95372049e-01,
        -9.36720082e-01, -8.17630180e-01,  1.88026639e+00],
       [-2.39077940e-01, -2.72872932e-01, -2.78070413e-01,
        -5.30121722e-01, -3.75772448e-01,  2.39753702e+00],
       [-3.39323419e+00, -2.41248290e+00, -2.99855551e+00,
         1.64922393e+00, -3.36667266e+00,  1.05550894e+02],
       [ 3.00000000e+00,  3.00000000e+00, -4.00000000e+00,
         3.00000000e+00,  3.00000000e+00,  2.00000000e+00],
       [ 3.00000000e+00,  3.00000000e+00,  3.00000000e+00,
        -4.00000000e+00,  3.00000000e+00,  2.00000000e+02],
       [ 3.00000000e+00, -4.00000000e+00, -4.00000000e+00,
         3.00000000e+00,  3.00000000e+00,  2.00000000e+00],
       [ 3.00000000e+00, -7.0726

In [33]:
tf.squeeze(tf.argmin(
    ask_tell.to_result().try_get_final_dataset().observations.numpy()
))

<tf.Tensor: shape=(), dtype=int64, numpy=11>

In [34]:
ask_tell.to_result().try_get_final_dataset().observations[11]

<tf.Tensor: shape=(1,), dtype=float64, numpy=array([-38.5922183])>

In [36]:
ask_tell.to_result().try_get_final_dataset().query_points[11].numpy()

array([-0.87547018, -1.80792785, -4.        ,  3.        , -1.38534975,
        2.        ])

In [29]:
initial_data

Dataset(query_points=array([[-0.25657046, -0.25657046, -0.25657046, -0.80367298, -0.80367298,
         2.21102834],
       [-0.12352336, -0.24261326, -0.29537205, -0.93672008, -0.81763018,
         1.88026639],
       [-0.23907794, -0.27287293, -0.27807041, -0.53012172, -0.37577245,
         2.39753702]]), observations=array([[ 1.18870531],
       [ 0.06914924],
       [-1.25785455]]))

In [37]:
normalize_backward(
    ask_tell.to_result(
    ).try_get_final_dataset().query_points[11].numpy()[0:5],
    design_matrix_mean,
    design_matrix_std
)

array([ -2.51506083,  -9.30584763, -25.26999823,  25.7087317 ,
        -6.22834836])

In [38]:
normalize_backward(
    ask_tell.to_result().try_get_final_dataset().query_points[11].numpy()[5],
    output_vals_mean,
    output_vals_std
)

198.40448355536665

In [45]:
sim.group_sequential_designs(
    upper_bounds = [-2.51506083,  -9.30584763, -25.26999823],
    lower_bounds = [25.7087317, -6.22834836, -25.26999823],
    n_patients = 198.40448355536665,
    alt_hypothesis = important_diff_delta,
    variance = assumed_variance
)

(            futility_null  efficacy_null  futility_alt  efficacy_alt
 analysis_1   1.000000e+00   9.940494e-01  1.000000e+00  1.000000e+00
 analysis_2  -6.036283e-13  -9.940494e-01 -6.347347e-89 -1.000000e+00
 analysis_3   0.000000e+00   6.035808e-13  0.000000e+00  1.425634e-49,
 6.035807986083234e-13,
 1.4256336483612423e-49,
 5.684341886080802e-14)

In [42]:
fp.feasibility_penalty(
    mu = mu,
    power=target_power,
    alpha=target_alpha,
    beta_prime=0,
    alpha_prime=0
)

-0.0

# The feasibility penalty might be broken.